In [2]:
!pip install pyspark

In [3]:
# --- CELL 1: Setup and Initialization ---
import re
import shutil
import os
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, length, split
from pyspark.sql.types import StringType

# Initialize Spark for Kaggle's CPU-Only Environment (takes advantage of 30GB RAM)
spark = SparkSession.builder \
    .appName("Arxiv_LLM_Data_Prep_Final") \
    .master("local[4]") \
    .config("spark.driver.memory", "24g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print("PySpark Engine Initialized!")

# --- CELL 2: Data Loading & Domain Filtering ---
# Load the raw arXiv JSON dataset
raw_data_path = "/kaggle/input/datasets/organizations/Cornell-University/arxiv/arxiv-metadata-oai-snapshot.json"
df = spark.read.json(raw_data_path)

# Filter for AI, Machine Learning, and Computer Vision papers
ml_papers = df.filter(col("categories").rlike("cs.AI|cs.LG|cs.CV"))

# Keep only the necessary columns to save memory
processed_df = ml_papers.select("id", "title", "abstract", "categories")

print("\n--- BEFORE CLEANING: Raw Academic Text ---")
# Display a 3-row sample of the raw, messy text
display(processed_df.select("id", "abstract").limit(3).toPandas())

# --- CELL 3: The Distributed Cleaning Engine ---
def clean_academic_text(text):
    if not text: 
        return ""
    
    # Remove newline characters
    text = text.replace('\n', ' ')
    # Strip out inline LaTeX equations
    text = re.sub(r'\$.*?\$', '', text)
    # Strip out complex LaTeX blocks 
    text = re.sub(r'\\begin\{.*?\}.*?\\end\{.*?\}', '', text)
    # Remove citation brackets 
    text = re.sub(r'\[\d+(,\s*\d+)*\]', '', text)
    # Clean up double spaces left behind by removals
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

# Register the Python function as a distributed PySpark UDF
clean_udf = udf(clean_academic_text, StringType())

# Apply the cleaning function
final_df = processed_df.withColumn("clean_abstract", clean_udf(col("abstract")))

# Calculate how much noise was removed for analytics
analytics_df = final_df.withColumn("raw_length", length(col("abstract"))) \
                       .withColumn("clean_length", length(col("clean_abstract"))) \
                       .withColumn("chars_removed", col("raw_length") - col("clean_length"))

print("\n--- AFTER CLEANING: Pristine Text ---")
# Display the exact same 3 rows to show the transformation and characters removed
display(analytics_df.select("id", "clean_abstract", "chars_removed").limit(3).toPandas())

print("\n--- PIPELINE IMPACT SUMMARY ---")
total_papers = analytics_df.count()
print(f"Total Papers Processed: {total_papers:,}")

# --- CELL 4: Partitioned Export & Zipping ---
print("\nExtracting primary category for folder partitioning...")
# Extract the primary category (the first category listed) for folder structuring
final_df = final_df.withColumn("primary_category", split(col("categories"), " ").getItem(0))

output_folder = "/kaggle/working/clean_ml_papers_partitioned.parquet"
print(f"Saving partitioned data to {output_folder} (This might take a minute)...")

# Save in "Folder Style" using partitionBy
final_df.select("id", "title", "clean_abstract", "primary_category") \
    .write \
    .mode("overwrite") \
    .partitionBy("primary_category") \
    .parquet(output_folder)

# Zip the Folder for easy download to your Windows PC
zip_path = "/kaggle/working/clean_academic_dataset"
print(f"Zipping the folder to {zip_path}.zip for download...")
shutil.make_archive(zip_path, 'zip', output_folder)

print("\nProcess Complete! You can now download the .zip file from the Kaggle Output panel.")

# --- CELL 5: Safe Shutdown ---
# Safely stop Spark NOW that everything is completely finished
spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/15 10:34:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


PySpark Engine Initialized!



--- BEFORE CLEANING: Raw Academic Text ---


,id,abstract
0,0704.0047,The intelligent acoustic emission locator is...
1,0704.0050,Part I describes an intelligent acoustic emi...
2,0704.0304,This paper discusses the benefits of describ...



--- AFTER CLEANING: Pristine Text ---


,id,clean_abstract,chars_removed
0,0704.0047,The intelligent acoustic emission locator is d...,5
1,0704.0050,Part I describes an intelligent acoustic emiss...,5
2,0704.0304,This paper discusses the benefits of describin...,5



--- PIPELINE IMPACT SUMMARY ---


Total Papers Processed: 492,020

Extracting primary category for folder partitioning...
Saving partitioned data to /kaggle/working/clean_ml_papers_partitioned.parquet (This might take a minute)...


Zipping the folder to /kaggle/working/clean_academic_dataset.zip for download...

Process Complete! You can now download the .zip file from the Kaggle Output panel.


In [4]:
from IPython.display import FileLink

# This generates a clickable HTML link right below the cell
FileLink(r'clean_academic_dataset.zip')

/kaggle/working/clean_academic_dataset.zip